# ocean: 3D

3D ocean data

**coordinate: tavg-ol-hxy-sea**
- tavg: time average
- ol: ocean levels
- hxy: horizontal curvlinear grids
- sea: ocean domain

In [ ]:
## Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cmocean
import sys
import os
import glob

sys.path.append(os.getcwd())
from utils import load_grid_vertex, read_variables, read_compound_name

In [ ]:
# parameters for the cmorized data
cmorout=''                  # root for cmorized data, e.g., '/scratch/$USER/cmorout'
source_id      = ''         # model name, e.g., 'NorESM3-LM'
experiment_id  = ''         # experiment name, e.g., 'historical', 'ssp585', 'piControl'
variant_label  = ''         # variant label, e.g., 'r1i1p1f1'
grid_label     = ''         # grid label, e.g., 'gn', 'gr', 'g999'
version        = ''         # version, e.g., 'v20260601'

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load grid
grid_file = 'data/grid.nc'
lat, lon, clat, clon = load_grid_vertex(grid_file)
with xr.open_dataset(grid_file) as ds:
    pmask = ds['pmask']
    parea = ds['parea']

parea = parea.rename({'y': 'j', 'x': 'i'})
pmask = pmask.rename({'y': 'j', 'x': 'i'})
# load methods for plotting and set defaults
methods =read_variables('data/methods.txt')
#print(methods.keys())

**List of data to be validated**

In [ ]:
# load compound names

cnames = read_compound_name()
# examples of compound names:
cnames = ['ocean.thetao.tavg-ol-hxy-sea.mon.glb', 'ocean.masscello.tavg-ol-hxy-sea.mon.glb']
for cname in cnames:
    print(cname)


In [ ]:
# loop through compound names and plot
for cname in cnames:
    mth_vert = 'mean'
    mth_ts = 'mean'
    mth_cmap = 'mpl.colormaps["viridis"]'
    if cname not in methods.keys():
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:
            if 'vertical' in methods[cname].keys():
                mth_vert = methods[cname]['vertical']

            if 'timeseries' in methods[cname].keys():
                mth_ts = methods[cname]['timeseries']

            if 'cmap' in methods[cname].keys():
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    grid = 'g999'

    if realm != 'ocean' or coord != 'tavg-ol-hxy-sea':
        continue

    data_file = var+'_'+coord+'_*_'+grid+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'

    if not glob.glob(os.path.join(data_path, data_file)):
        continue

    #with xr.open_mfdataset(os.path.join(data_path, data_file)) as ds:
    data_file = glob.glob(os.path.join(data_path, data_file))[0]
    with xr.open_dataset(os.path.join(data_path, data_file)) as ds:
        if var in ds:
            data = ds[var]
        else:
            continue

    if 'lev' not in data.dims:
        print(f"{cname} does not have vertical dimension 'lev' , skipping vertical aggregation.")
        continue
   
    if mth_vert == 'mean':
        data2d = data.mean(dim='lev',keep_attrs=True).where(pmask == 1)
    elif mth_vert == 'sum':
        data2d = data.sum(dim='lev',keep_attrs=True).where(pmask == 1)
    else:
        raise ValueError(f'Unsupported vertical aggregation method: {mth_vert}')

    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data2d.long_name} ({data2d.units})')
    print(f'vertical aggregation method for 3D->2D plot: {mth_vert}')
    print(f'timeseries aggregation method for 3D->1D plot: {mth_ts}')

    #ax1 = fig.add_subplot(1, 2, 1, projection=proj)
    #fig, (ax1, ax2) = plt.subplots( nrows=1, ncols=2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 1]}, subplot_kw={'projection': proj})

    fig = plt.figure(figsize=(16, 3), dpi=96)
    gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[2, 1]) 
    proj = ccrs.PlateCarree(central_longitude=90.0)

    # plot 2D map
    ax1 = fig.add_subplot(gs[0], projection=proj)
    # Plot the mesh
    pm = ax1.pcolormesh(lon, lat, data2d.mean(dim='time'), cmap=eval(mth_cmap), norm=None,
                        transform=ccrs.PlateCarree(), shading='auto', rasterized=True)

    # Add map features
    #ax.stock_img()
    ax1.add_feature(cfeature.LAND, facecolor='lightgray')

    gl = ax1.gridlines(ylocs=range(-90, 90, 30), draw_labels=True)
    gl.ylocator = mpl.ticker.FixedLocator(range(-90,90,30))

    # Add colorbar
    cb = plt.colorbar(pm, ax=ax1, fraction=0.4, shrink=0.8, label='[yr]')
    cb.set_label(label=data.units, size=14)
    cb.ax.tick_params(labelsize=12)
    plt.tight_layout()
        
    #fig, ax = plot_map2d(lon, lat, data2d.mean(dim='time'), proj='PlateCarree', norm=None, cmap=eval(mth_cmap))
    ax1.coastlines(resolution='110m')
    ax1.add_feature(cfeature.LAND, facecolor='lightgray')
    plt.title(data2d.long_name)

    # plot 1D timeseries
    if mth_ts == 'mean':
        data1d = (data2d*parea).mean(dim=['j', 'i'], keep_attrs=True)
    elif mth_ts == 'sum':
        data1d = (data2d*parea).sum(dim=['j', 'i'], keep_attrs=True)
    else:
        raise ValueError(f'Unsupported timeseries method: {mth_ts}')
   
    data1d.attrs = data2d.attrs.copy()

    ax2 = fig.add_subplot(gs[1])
    if data1d.sizes["time"] < 2:
        ax2.text(0.1, 0.8, (f"global {mth_ts} value: {data1d.values[0]} {data1d.units}"))
        ax2.text(0.1, 0.6, ("less than 2 time steps."))
        ax2.text(0.1, 0.4, ("skipping timeseries plot."))
    else:
        data1d.plot(ax=ax2)

    plt.title(data1d.long_name)
    plt.show()

    del data, data2d, data1d
    del ax1, ax2, pm, cb, gl, fig